In [5]:
import os
import re

def remove_comments_from_code(code: str, extension: str) -> str:
    """Remove comments from code depending on file type."""
    # Java / Kotlin / C / C++ / JS / Gradle
    if extension in ('.java', '.kt', '.js', '.c', '.cpp', '.h', '.gradle'):
        pattern = r'(?m)^\s*//.*$|/\*[\s\S]*?\*/'
        return re.sub(pattern, '', code)

    # Python
    elif extension == '.py':
        pattern = r'(?m)^\s*#.*$|"""[\s\S]*?"""|\'\'\'[\s\S]*?\'\'\''
        return re.sub(pattern, '', code)

    # XML / HTML
    elif extension in ('.xml', '.html'):
        pattern = r'<!--[\s\S]*?-->'
        return re.sub(pattern, '', code)

    # JSON / text
    else:
        return code


def print_files_without_comments(
    root_dir,
    ignore_dirs=None,
    extensions=None
):
    if extensions is None:
        extensions = ('.kt', '.java', '.xml', '.gradle', '.py', '.json', '.txt')

    if ignore_dirs is None:
        ignore_dirs = []

    ignore_dirs = set(ignore_dirs)

    for root, dirs, files in os.walk(root_dir):
        # 🔹 Modify dirs in-place to skip ignored folders
        dirs[:] = [d for d in dirs if d not in ignore_dirs]

        for file in files:
            ext = os.path.splitext(file)[1]
            if ext in extensions:
                file_path = os.path.join(root, file)
                print(f"\n--- {file_path} ---")
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        code = f.read()
                        cleaned_code = remove_comments_from_code(code, ext)
                        for line in cleaned_code.splitlines():
                            if line.strip():
                                print(line)
                except Exception as e:
                    print(f"[Error reading {file_path}: {e}]")


# ---------------- Example usage ----------------

ignore = ['ui', 'util']

directory_path = r"src/main/java/com/samsung/health/mobile"
print_files_without_comments(directory_path, ignore_dirs=ignore)

# directory_path = r"src/test/java/com/samsung/health/mobile"
# print_files_without_comments(directory_path, ignore_dirs=ignore)



--- src/main/java/com/samsung/health/mobile\DataReceiverService.kt ---
package com.samsung.health.mobile
import android.util.Log
import com.google.android.gms.wearable.MessageEvent
import com.google.android.gms.wearable.WearableListenerService
import com.samsung.health.mobile.util.PerformanceMonitor
import dagger.hilt.android.AndroidEntryPoint
import kotlinx.coroutines.CoroutineScope
import kotlinx.coroutines.Dispatchers
import kotlinx.coroutines.SupervisorJob
import kotlinx.coroutines.launch
import kotlinx.serialization.json.Json
import javax.inject.Inject
@AndroidEntryPoint
class DataReceiverService : WearableListenerService() {
    @Inject lateinit var storageManager: StorageManager
    @Inject lateinit var repository: HealthDataRepository // <--- INJECT REPO
    private val scope = CoroutineScope(SupervisorJob() + Dispatchers.IO)
    override fun onMessageReceived(event: MessageEvent) {
        if (event.path == "/raw_data_stream") {
            val packetSize = event.data.size.to